# USM v2 — Dimensionality Sweep: Hyperbolic vs Euclidean

Trains **both** geometries at `d = [32, 64, 128, 256, 512]` on the same data and architecture, proving that **hyperbolic embeddings achieve equivalent quality in far fewer dimensions**.

**Before running:**
1. **Settings → Accelerator → GPU T4 x2**
2. **Settings → Internet → ON**

Backbone embeddings are pre-cached once and reused across all runs.

**Estimated time:** ~2.5–3 hours on T4 x2.

In [ ]:
!pip install -q geoopt sentence-transformers transformers datasets umap-learn torchvision
print('Dependencies installed.')


## Library (all usm_v2 code inlined)

In [ ]:
# =============================================================================
# USM v2 — inlined library (single-notebook build for Kaggle)
# =============================================================================
import os, re, random, time, math
from collections import defaultdict, Counter
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import geoopt


# ----- config.py -----
import torch
from dataclasses import dataclass, field
from typing import Optional


@dataclass
class USMConfig:
    seed: int = 42

    # --- Manifold (per-relation scheduled curvature) ---
    c_init: float = 0.001
    c_min: float = 0.0001
    c_max: float = 10.0
    # Per-relation targets: IS_A, CAUSES, PART_OF, SIMILAR_TO, ANTONYM, CAPABLE_OF
    c_targets: tuple = (0.05, 0.01, 0.03, 0.003, 0.003, 0.01)
    c_warmup_start: int = 10
    c_warmup_end: int = 40
    learnable_curvature: bool = False

    # --- Dimensions ---
    d: int = 1024
    d_backbone: int = 384  # MiniLM output
    d_clip: int = 768

    # --- Backbones ---
    text_backbone: str = "paraphrase-multilingual-MiniLM-L12-v2"
    clip_model: str = "openai/clip-vit-large-patch14"

    # --- Relations ---
    relations: tuple = ("IS_A", "CAUSES", "PART_OF", "SIMILAR_TO", "ANTONYM", "CAPABLE_OF")
    n_relations: int = 6

    # --- Training Phase 1 ---
    n_epochs_p1: int = 60
    batch_size: int = 256
    lr_p1: float = 5e-4
    weight_decay: float = 1e-4
    grad_clip_norm: float = 1.0

    # --- Training Phase 2 ---
    n_epochs_p2: int = 60
    lr_p2_vis: float = 3e-4
    lr_p2_text: float = 1e-5
    p2_batch: int = 1024

    # --- Data ---
    max_cn_triples: int = 400_000
    max_cl_per_lang: int = 20_000
    max_snli_pairs: int = 20_000

    # --- Gradient control ---
    burnin_epochs: int = 0
    transition_epochs: int = 0
    use_riemannian_clipping: bool = True

    # --- Curriculum ---
    curriculum_enabled: bool = True
    curriculum_full_data_pct: float = 0.7
    hard_neg_max_ratio: float = 0.8

    # --- Loss weights (base values, curriculum may modulate) ---
    w_rel: float = 1.0
    w_comp: float = 0.1
    w_cl: float = 1.0
    w_ent: float = 0.5
    w_hier: float = 0.5

    # --- Loss margins ---
    margin_rel: float = 2.0
    margin_hier: float = 1.0
    delta_contradiction: float = 2.0

    # --- Infrastructure ---
    ckpt_dir: str = "/tmp/usm_v2_checkpoints"
    ckpt_every: int = 5
    encode_batch: int = 512
    grad_accum: int = 2
    max_clip_images: int = 50_000

    # --- Run mode ---
    # True = fast Colab smoke test (~5–10 min). False = training scale.
    validation_mode: bool = False

    # --- Computed at runtime ---
    device: Optional[torch.device] = field(default=None, repr=False)
    backbone_device: Optional[torch.device] = field(default=None, repr=False)
    use_bf16: bool = False
    large_gpu: bool = False
    n_gpus: int = 1

    def __post_init__(self):
        if self.device is None:
            self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        if torch.cuda.is_available():
            self.n_gpus = torch.cuda.device_count()
            vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
            self.large_gpu = vram_gb >= 24
            self.use_bf16 = vram_gb >= 38
        else:
            self.n_gpus = 0
            self.large_gpu = False
            self.use_bf16 = False

        if self.backbone_device is None:
            if self.n_gpus >= 2:
                self.backbone_device = torch.device("cuda:1")
            else:
                self.backbone_device = self.device

        if self.validation_mode:
            self.apply_validation_scale()
        elif self.n_gpus >= 2 and not self.large_gpu:
            self._apply_dual_t4_scale()
        elif not self.large_gpu:
            self._apply_medium_scale()

    def _apply_dual_t4_scale(self):
        """Optimized for Kaggle T4 x2: backbone on GPU 1, training on GPU 0."""
        self.d = 512
        self.d_clip = 512
        self.clip_model = "openai/clip-vit-base-patch32"
        self.n_epochs_p1 = 50
        self.n_epochs_p2 = 40
        self.batch_size = 512
        self.lr_p1 = 3e-4
        self.p2_batch = 2048
        self.max_cn_triples = 300_000
        self.max_cl_per_lang = 30_000
        self.max_snli_pairs = 30_000
        self.max_clip_images = 50_000
        self.encode_batch = 512
        self.grad_accum = 1

    def _apply_medium_scale(self):
        """Downscale for single T4 / consumer GPU (12-16 GB)."""
        self.d = 512
        self.d_clip = 512
        self.clip_model = "openai/clip-vit-base-patch32"
        self.n_epochs_p1 = 30
        self.n_epochs_p2 = 30
        self.batch_size = 128
        self.max_cn_triples = 100_000
        self.encode_batch = 256
        self.grad_accum = 1

    def apply_validation_scale(self):
        """Fast smoke test: verify pipeline, not benchmark quality (~5–10 min on T4)."""
        self.d = 256
        self.d_clip = 256
        self.clip_model = "openai/clip-vit-base-patch32"
        self.n_epochs_p1 = 5
        self.n_epochs_p2 = 2
        self.batch_size = 64
        self.p2_batch = 256
        self.max_cn_triples = 3_000
        self.max_cl_per_lang = 300
        self.max_snli_pairs = 300
        self.max_clip_images = 4_000
        self.encode_batch = 128
        self.burnin_epochs = 1
        self.transition_epochs = 1
        self.curriculum_enabled = False
        self.ckpt_every = 999
        self.grad_accum = 1

# ----- manifold.py -----
import torch
import torch.nn as nn
import geoopt

EPS = 1e-5


class LearnablePoincareBall(nn.Module):
    """
    Poincaré ball with per-relation learnable curvature.

    Each relation type gets its own curvature c_r. The base curvature
    (used for projection) is max(c_r) so all points fit in every
    relation-specific ball. This lets IS_A develop strong curvature for
    tree structure while SIMILAR_TO stays near-Euclidean.
    """

    def __init__(self, c_init: float = 0.001, c_min: float = 0.0001, c_max: float = 10.0,
                 learnable: bool = True, n_relations: int = 6):
        super().__init__()
        self.c_min = c_min
        self.c_max = c_max
        self.n_relations = n_relations
        if learnable:
            self._c_params = nn.Parameter(torch.full((n_relations,), float(c_init)))
        else:
            self.register_buffer("_c_params", torch.full((n_relations,), float(c_init)))

    @property
    def c(self) -> torch.Tensor:
        """Base curvature for projection (max over all relations)."""
        return torch.clamp(self._c_params.max(), self.c_min, self.c_max)

    def c_rel(self, rel_idx: int) -> torch.Tensor:
        """Curvature for a specific relation index (scalar)."""
        return torch.clamp(self._c_params[rel_idx], self.c_min, self.c_max)

    def effective_c(self, scale: float = 1.0) -> torch.Tensor:
        """Base curvature modulated by burn-in scale in [0, 1]."""
        if scale <= 0.0:
            return torch.tensor(0.0, device=self._c_params.device)
        return self.c * scale

    def effective_c_rel(self, rel_idx: int, scale: float = 1.0) -> torch.Tensor:
        """Per-relation curvature modulated by burn-in scale."""
        if scale <= 0.0:
            return torch.tensor(0.0, device=self._c_params.device)
        return self.c_rel(rel_idx) * scale

    @property
    def ball(self) -> geoopt.PoincareBall:
        return geoopt.PoincareBall(c=self.c)

    def max_norm(self, c: torch.Tensor) -> torch.Tensor:
        """Maximum allowed norm for points: 1/sqrt(c) - eps."""
        safe_c = c.clamp(min=EPS)
        return (1.0 / safe_c.sqrt()) - EPS


def clamp_to_ball(z: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
    """Project z inside the Poincaré ball of curvature c."""
    safe_c = c.clamp(min=EPS)
    max_n = (1.0 / safe_c.sqrt()) - EPS
    norms = z.norm(dim=-1, keepdim=True).clamp(min=1e-10)
    scale = torch.where(norms >= max_n, max_n / norms, torch.ones_like(norms))
    return z * scale


def conformal_factor(z: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
    """
    Conformal factor lambda_x = 2 / (1 - c * ||x||^2).
    Shape: same as z but with last dim squeezed.
    """
    safe_c = c.clamp(min=EPS)
    norm_sq = z.pow(2).sum(dim=-1, keepdim=True)
    return 2.0 / (1.0 - safe_c * norm_sq).clamp(min=EPS)


# ---------------------------------------------------------------------------
# Hyperbolic primitives parameterized by curvature c
# ---------------------------------------------------------------------------

def _mobius_add(x: torch.Tensor, y: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
    """Möbius addition in the Poincaré ball with curvature c."""
    x_sq = (x * x).sum(dim=-1, keepdim=True)
    y_sq = (y * y).sum(dim=-1, keepdim=True)
    xy = (x * y).sum(dim=-1, keepdim=True)
    num = (1 + 2 * c * xy + c * y_sq) * x + (1 - c * x_sq) * y
    denom = (1 + 2 * c * xy + c * c * x_sq * y_sq).clamp(min=EPS)
    return num / denom


def expmap0(v: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
    """Exponential map from the origin."""
    safe_c = c.clamp(min=EPS)
    sqrt_c = safe_c.sqrt()
    v_norm = v.norm(dim=-1, keepdim=True).clamp(min=1e-10)
    return clamp_to_ball(
        torch.tanh(sqrt_c * v_norm) * v / (sqrt_c * v_norm),
        c
    )


def logmap0(z: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
    """Logarithmic map to the origin (inverse of expmap0)."""
    z = clamp_to_ball(z, c)
    safe_c = c.clamp(min=EPS)
    sqrt_c = safe_c.sqrt()
    z_norm = z.norm(dim=-1, keepdim=True).clamp(min=1e-10)
    return torch.atanh(sqrt_c * z_norm).clamp(max=10.0) * z / (sqrt_c * z_norm)


def expmap(x: torch.Tensor, v: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
    """Exponential map from point x in direction v."""
    safe_c = c.clamp(min=EPS)
    sqrt_c = safe_c.sqrt()
    x = clamp_to_ball(x, c)
    lam = conformal_factor(x, c)
    v_norm = v.norm(dim=-1, keepdim=True).clamp(min=1e-10)
    second = torch.tanh(sqrt_c * lam * v_norm / 2.0) * v / (sqrt_c * v_norm)
    return clamp_to_ball(_mobius_add(x, second, c), c)


def poincare_dist(z1: torch.Tensor, z2: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
    """Geodesic distance between z1 and z2 on the Poincaré ball."""
    safe_c = c.clamp(min=EPS)
    sqrt_c = safe_c.sqrt()
    z1 = clamp_to_ball(z1, c)
    z2 = clamp_to_ball(z2, c)
    diff_sq = (z1 - z2).pow(2).sum(dim=-1)
    n1_sq = z1.pow(2).sum(dim=-1)
    n2_sq = z2.pow(2).sum(dim=-1)
    denom = ((1.0 - safe_c * n1_sq) * (1.0 - safe_c * n2_sq)).clamp(min=EPS)
    arg = 1.0 + 2.0 * safe_c * diff_sq / denom
    return (1.0 / sqrt_c) * torch.acosh(arg.clamp(min=1.0 + 1e-6))


def poincare_cdist(z1: torch.Tensor, z2: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
    """Batched pairwise geodesic distance matrix [B1, B2]."""
    safe_c = c.clamp(min=EPS)
    sqrt_c = safe_c.sqrt()
    z1 = clamp_to_ball(z1, c)
    z2 = clamp_to_ball(z2, c)
    n1_sq = z1.pow(2).sum(dim=-1, keepdim=True)
    n2_sq = z2.pow(2).sum(dim=-1, keepdim=True).T
    dot = z1 @ z2.T
    diff_sq = (n1_sq + n2_sq - 2.0 * dot).clamp(min=0.0)
    denom = ((1.0 - safe_c * n1_sq) * (1.0 - safe_c * n2_sq)).clamp(min=EPS)
    arg = 1.0 + 2.0 * safe_c * diff_sq / denom
    return (1.0 / sqrt_c) * torch.acosh(arg.clamp(min=1.0 + 1e-6))


def dist0(z: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
    """Geodesic distance from the origin."""
    z = clamp_to_ball(z, c)
    safe_c = c.clamp(min=EPS)
    sqrt_c = safe_c.sqrt()
    n_sq = z.pow(2).sum(dim=-1)
    arg = 1.0 + 2.0 * safe_c * n_sq / (1.0 - safe_c * n_sq).clamp(min=EPS)
    return (1.0 / sqrt_c) * torch.acosh(arg.clamp(min=1.0))


# ---------------------------------------------------------------------------
# Euclidean equivalents (baseline)
# ---------------------------------------------------------------------------

def eucl_dist(z1: torch.Tensor, z2: torch.Tensor) -> torch.Tensor:
    return (z1 - z2).pow(2).sum(dim=-1).sqrt()


def eucl_cdist(z1: torch.Tensor, z2: torch.Tensor) -> torch.Tensor:
    return torch.cdist(z1, z2, p=2)

# ----- gradient_control.py -----
import torch


# ---------------------------------------------------------------------------
# 1. Conformal-factor gradient scaling
# ---------------------------------------------------------------------------

def scale_riemannian_grads(params, c: torch.Tensor):
    """
    Rescale Euclidean gradients of manifold parameters by the inverse
    squared conformal factor.

    In the Poincaré ball, the Riemannian metric at x is g_x = λ_x² · I,
    so the Riemannian gradient is (1/λ²) times the Euclidean gradient.
    Without this rescaling, points near the boundary receive explosively
    large effective updates — the root cause of radial collapse.
    """
    if c.item() < EPS:
        return

    for p in params:
        if p.grad is None or p.dim() < 1:
            continue
        lam = conformal_factor(p.data, c)  # [..., 1]
        p.grad.data.div_(lam.pow(2).clamp(max=1e6))


# ---------------------------------------------------------------------------
# 2. Riemannian-aware gradient clipping
# ---------------------------------------------------------------------------

def clip_riemannian_grad_norm(params, max_norm: float, c: torch.Tensor):
    """
    Clip gradients using Riemannian norm rather than Euclidean norm.

    The Riemannian norm of a tangent vector v at point x is ||v||_x = ||v|| / λ_x.
    Clipping in this metric prevents over-correction near the boundary while
    allowing healthy updates near the origin.
    """
    if c.item() < EPS:
        torch.nn.utils.clip_grad_norm_(params, max_norm)
        return

    total_norm_sq = 0.0
    grads = []
    for p in params:
        if p.grad is None:
            continue
        if p.dim() >= 1:
            lam = conformal_factor(p.data, c)
            riem_grad = p.grad.data / lam
            riem_norm = riem_grad.norm()
        else:
            riem_norm = p.grad.data.norm()
        total_norm_sq += riem_norm.item() ** 2
        grads.append(p.grad)

    total_norm = total_norm_sq ** 0.5
    clip_coef = max_norm / (total_norm + 1e-6)
    if clip_coef < 1.0:
        for g in grads:
            g.data.mul_(clip_coef)


# ---------------------------------------------------------------------------
# 3. Euclidean burn-in scheduler
# ---------------------------------------------------------------------------

class BurninScheduler:
    """
    Controls a smooth transition from Euclidean (c_eff=0) to full hyperbolic
    geometry over training.

    During burn-in epochs the curvature scale is 0 (pure Euclidean), then it
    linearly ramps to 1.0 over the transition window. This lets the model
    learn a reasonable initial layout before hyperbolic geometry amplifies
    gradients near the boundary.

    Inspired by Nickel & Kiela (2017) who found burn-in critical for
    Poincaré embedding stability.
    """

    def __init__(self, burnin_epochs: int = 10, transition_epochs: int = 5):
        self.burnin_epochs = burnin_epochs
        self.transition_epochs = transition_epochs

    def get_curvature_scale(self, epoch: int) -> float:
        """Returns curvature multiplier in [0, 1]."""
        if epoch < self.burnin_epochs:
            return 0.0
        elif epoch < self.burnin_epochs + self.transition_epochs:
            progress = (epoch - self.burnin_epochs) / max(self.transition_epochs, 1)
            return min(progress, 1.0)
        return 1.0

    def is_hyperbolic(self, epoch: int) -> bool:
        return self.get_curvature_scale(epoch) > 0.0

# ----- operators.py -----
import torch
import torch.nn as nn
from typing import Optional



RELATIONS = ("IS_A", "CAUSES", "PART_OF", "SIMILAR_TO", "ANTONYM", "CAPABLE_OF")
REL2IDX = {r: i for i, r in enumerate(RELATIONS)}
N_RELATIONS = len(RELATIONS)

_SYMW = {
    "IS_A": 0.0, "CAUSES": 0.0, "PART_OF": 0.0,
    "SIMILAR_TO": 1.0, "ANTONYM": 0.1, "CAPABLE_OF": 0.0,
}
SYM_WEIGHTS = torch.tensor([_SYMW[r] for r in RELATIONS], dtype=torch.float32)


class CompositionalOperator(nn.Module):
    """
    Typed relational composition: z_A ⊕_r z_B.

    Operates in tangent space at origin (via logmap0), applies a bilinear
    relation-specific transform, then maps back to the ball (via expmap0).
    Symmetry weights blend U toward W for symmetric relations.
    """

    def __init__(self, manifold: LearnablePoincareBall, d: int = 1024,
                 n_rel: int = N_RELATIONS, hyperbolic: bool = True):
        super().__init__()
        self.d = d
        self.hyperbolic = hyperbolic
        self.manifold = manifold

        init = lambda: (
            torch.eye(d).unsqueeze(0).repeat(n_rel, 1, 1)
            + 0.01 * torch.randn(n_rel, d, d)
        )
        self.W = nn.Parameter(init())
        self.U_raw = nn.Parameter(init())
        self.V = nn.Parameter(0.01 * torch.randn(n_rel, d, d))
        self.b = nn.Parameter(torch.zeros(n_rel, d))
        self.register_buffer("sym_w", SYM_WEIGHTS)

    def effective_U(self):
        s = self.sym_w.view(-1, 1, 1)
        return s * self.W + (1.0 - s) * self.U_raw

    def forward(self, z_A: torch.Tensor, rel_idx: torch.Tensor,
                z_B: torch.Tensor, c: Optional[torch.Tensor] = None) -> torch.Tensor:
        if c is None:
            c = self.manifold.c

        if self.hyperbolic and c.item() > EPS:
            a, b = logmap0(z_A, c), logmap0(z_B, c)
        else:
            a, b = z_A, z_B

        U_eff = self.effective_U()
        W_r = self.W[rel_idx]
        U_r = U_eff[rel_idx]
        V_r = self.V[rel_idx]
        b_r = self.b[rel_idx]

        out = torch.tanh(
            torch.einsum("bij,bj->bi", W_r, a)
            + torch.einsum("bij,bj->bi", U_r, b)
            + torch.einsum("bij,bj->bi", V_r, a * b)
            + b_r
        )

        if self.hyperbolic and c.item() > EPS:
            return clamp_to_ball(expmap0(out, c), c)
        return out


class RelationMaps(nn.Module):
    """
    TransE-style relation translation: z_h --r--> z_pred.

    Per-relation linear map in tangent space.
    """

    def __init__(self, manifold: LearnablePoincareBall, d: int = 1024,
                 n_rel: int = N_RELATIONS, hyperbolic: bool = True):
        super().__init__()
        self.hyperbolic = hyperbolic
        self.manifold = manifold
        self.W = nn.Parameter(
            torch.eye(d).unsqueeze(0).repeat(n_rel, 1, 1)
            + 0.01 * torch.randn(n_rel, d, d)
        )
        self.b = nn.Parameter(torch.zeros(n_rel, d))

    def forward(self, z_h: torch.Tensor, rel_idx: torch.Tensor,
                c: Optional[torch.Tensor] = None) -> torch.Tensor:
        if c is None:
            c = self.manifold.c

        if self.hyperbolic and c.item() > EPS:
            h = logmap0(z_h, c)
        else:
            h = z_h

        W_r = self.W[rel_idx]
        b_r = self.b[rel_idx]
        out = torch.einsum("bij,bj->bi", W_r, h) + b_r

        if self.hyperbolic and c.item() > EPS:
            return clamp_to_ball(expmap0(out, c), c)
        return out

# ----- losses.py -----
import torch
import torch.nn.functional as F
from typing import Optional



def _is_hyperbolic(c: torch.Tensor) -> bool:
    return c.item() > EPS


def loss_cl(z_src: torch.Tensor, z_tgt: torch.Tensor, c: torch.Tensor,
            tau: float = 1.0) -> torch.Tensor:
    """Symmetric InfoNCE with negative geodesic (or Euclidean) distance as similarity."""
    B = z_src.shape[0]
    if B == 0:
        return z_src.new_zeros(())

    cdist_fn = poincare_cdist if _is_hyperbolic(c) else eucl_cdist
    if _is_hyperbolic(c):
        dists = cdist_fn(z_src, z_tgt, c)
    else:
        dists = cdist_fn(z_src, z_tgt)

    sim = -dists / tau
    labels = torch.arange(B, device=z_src.device)
    return 0.5 * (F.cross_entropy(sim, labels) + F.cross_entropy(sim.T, labels))


def loss_rel(z_pred: torch.Tensor, z_t: torch.Tensor, z_t_neg: torch.Tensor,
             c: torch.Tensor, margin: float = 2.0) -> torch.Tensor:
    """TransE-style margin loss."""
    if _is_hyperbolic(c):
        d_pos = poincare_dist(z_pred, z_t, c)
        d_neg = poincare_dist(z_pred, z_t_neg, c)
    else:
        d_pos = eucl_dist(z_pred, z_t)
        d_neg = eucl_dist(z_pred, z_t_neg)
    return F.relu(margin - d_neg + d_pos).mean()


def loss_entailment(z_A: torch.Tensor, z_B: torch.Tensor, labels: torch.Tensor,
                    c: torch.Tensor, delta_c: float = 2.0) -> torch.Tensor:
    """
    Partial order + contradiction margin loss from SNLI.
    Labels: 0 = entailment, 1 = neutral (ignored), 2 = contradiction.
    """
    total, n = z_A.new_zeros(()), 0

    ent_mask = (labels == 0)
    contra_mask = (labels == 2)

    if _is_hyperbolic(c):
        lm = lambda x: logmap0(x, c)
        dist_fn = lambda a, b: poincare_dist(a, b, c)
    else:
        lm = lambda x: x
        dist_fn = eucl_dist

    if ent_mask.any():
        a_e, b_e = lm(z_A[ent_mask]), lm(z_B[ent_mask])
        total = total + F.relu(b_e - a_e).pow(2).sum(-1).mean()
        n += 1

    if contra_mask.any():
        d_c = dist_fn(z_A[contra_mask], z_B[contra_mask])
        total = total + F.relu(delta_c - d_c).pow(2).mean()
        n += 1

    return total / max(n, 1)


def loss_crossmodal(z_text: torch.Tensor, z_image: torch.Tensor,
                    c: torch.Tensor, tau: float = 1.0) -> torch.Tensor:
    """Cross-modal InfoNCE: pull text-image pairs together."""
    return loss_cl(z_text, z_image, c, tau=tau)


def loss_hierarchy(z_fine: torch.Tensor, z_coarse: torch.Tensor,
                   c: torch.Tensor, margin: float = 1.0) -> torch.Tensor:
    """
    Fine-class embeddings should be deeper on the ball (higher depth) than coarse.
    Combines margin ranking + entailment proximity penalty.
    """
    if _is_hyperbolic(c):
        depth_fine = dist0(z_fine, c)
        depth_coarse = dist0(z_coarse, c)
    else:
        depth_fine = z_fine.norm(dim=-1)
        depth_coarse = z_coarse.norm(dim=-1)

    L_rank = F.margin_ranking_loss(
        depth_fine, depth_coarse,
        target=torch.ones_like(depth_fine),
        margin=margin,
    )

    if _is_hyperbolic(c):
        L_close = poincare_dist(z_fine, z_coarse, c).mean() * 0.1
    else:
        L_close = (z_fine - z_coarse).norm(dim=-1).mean() * 0.1

    return L_rank + L_close

# ----- encoders.py -----
import torch
import torch.nn as nn
import geoopt
from typing import List, Tuple, Optional



class ConceptEncoder(nn.Module):
    """
    Text concept encoder: frozen MiniLM backbone -> trainable projection -> Poincaré ball.
    """

    def __init__(self, manifold: LearnablePoincareBall, d_out: int = 1024,
                 backbone: str = "paraphrase-multilingual-MiniLM-L12-v2",
                 hyperbolic: bool = True, device: torch.device = torch.device("cpu"),
                 backbone_device: Optional[torch.device] = None):
        super().__init__()
        from sentence_transformers import SentenceTransformer

        self.hyperbolic = hyperbolic
        self.manifold = manifold
        self.device = device
        self.backbone_device = backbone_device or device

        self.backbone = SentenceTransformer(backbone, device=str(self.backbone_device))
        for p in self.backbone.parameters():
            p.requires_grad_(False)

        d_in = self.backbone.get_sentence_embedding_dimension()
        self.proj = nn.Sequential(
            nn.Linear(d_in, d_out),
            nn.LayerNorm(d_out),
        )

        if hyperbolic:
            self.mu0 = geoopt.ManifoldParameter(
                torch.zeros(d_out),
                manifold=geoopt.PoincareBall(c=1.0),
            )

    def to(self, *args, **kwargs):
        """Move trainable layers but keep frozen backbone on backbone_device."""
        super().to(*args, **kwargs)
        if hasattr(self, 'proj') and len(list(self.proj.parameters())) > 0:
            self.device = next(self.proj.parameters()).device
        self.backbone.to(self.backbone_device)
        return self

    @torch.no_grad()
    def encode_backbone(self, texts: List[str]) -> torch.Tensor:
        embs = self.backbone.encode(texts, convert_to_tensor=True, show_progress_bar=False)
        return embs.to(self.device).float()

    def project(self, bb_emb: torch.Tensor, c: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        Project backbone embeddings onto the manifold.

        Args:
            bb_emb: backbone embeddings [B, d_backbone]
            c: effective curvature (if None, uses manifold.c)
        """
        if c is None:
            c = self.manifold.c

        bb_emb = bb_emb.clone()
        v = self.proj(bb_emb)

        if self.hyperbolic and c.item() > EPS:
            mu0 = self.mu0.unsqueeze(0).expand(v.shape[0], -1)
            return clamp_to_ball(expmap(mu0, v, c), c)
        return v

    def forward(self, texts: List[str],
                c: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, torch.Tensor]:
        bb_emb = self.encode_backbone(texts)
        z = self.project(bb_emb, c=c)
        return z, bb_emb


class VisionEncoder(nn.Module):
    """
    Vision encoder: frozen CLIP backbone -> trainable projection -> Poincaré ball.
    """

    def __init__(self, manifold: LearnablePoincareBall, d_out: int = 1024,
                 clip_model: str = "openai/clip-vit-large-patch14",
                 d_clip: int = 768, hyperbolic: bool = True,
                 device: torch.device = torch.device("cpu"),
                 backbone_device: Optional[torch.device] = None):
        super().__init__()
        from transformers import CLIPModel

        self.hyperbolic = hyperbolic
        self.manifold = manifold
        self.device = device
        self.backbone_device = backbone_device or device

        self.clip = CLIPModel.from_pretrained(clip_model).to(self.backbone_device)
        self.clip.eval()
        for p in self.clip.parameters():
            p.requires_grad_(False)

        self.proj = nn.Sequential(
            nn.Linear(d_clip, d_out),
            nn.LayerNorm(d_out),
        )

        if hyperbolic:
            self.mu0 = geoopt.ManifoldParameter(
                torch.zeros(d_out),
                manifold=geoopt.PoincareBall(c=1.0),
            )

    def to(self, *args, **kwargs):
        """Move trainable layers but keep frozen CLIP on backbone_device."""
        super().to(*args, **kwargs)
        if hasattr(self, 'proj') and len(list(self.proj.parameters())) > 0:
            self.device = next(self.proj.parameters()).device
        self.clip.to(self.backbone_device)
        return self

    @torch.no_grad()
    def encode_images(self, pixel_values: torch.Tensor) -> torch.Tensor:
        vision_out = self.clip.vision_model(pixel_values=pixel_values.to(self.backbone_device))
        pooled = vision_out.pooler_output
        pooled = self.clip.visual_projection(pooled)
        return pooled.to(self.device).float()

    def project(self, clip_emb: torch.Tensor, c: Optional[torch.Tensor] = None) -> torch.Tensor:
        if c is None:
            c = self.manifold.c

        clip_emb = clip_emb.clone()
        v = self.proj(clip_emb)

        if self.hyperbolic and c.item() > EPS:
            mu0 = self.mu0.unsqueeze(0).expand(v.shape[0], -1)
            return clamp_to_ball(expmap(mu0, v, c), c)
        return v

    def forward(self, pixel_values: torch.Tensor,
                c: Optional[torch.Tensor] = None) -> torch.Tensor:
        clip_emb = self.encode_images(pixel_values)
        return self.project(clip_emb, c=c)

# ----- curriculum.py -----
import random
import torch
from collections import defaultdict
from typing import List, Tuple, Dict, Optional


# ---------------------------------------------------------------------------
# 1. Difficulty scoring
# ---------------------------------------------------------------------------

def compute_difficulty_scores(
    triples: List[Tuple[str, str, str]],
    vocab_list: List[str],
) -> Dict[str, float]:
    """
    Score each concept by its depth in the IS_A hierarchy.

    Root concepts (e.g., 'animal', 'object') get low scores (easy).
    Leaf concepts (e.g., 'golden retriever') get high scores (hard).
    Concepts not in the IS_A tree get a default mid-range score.

    Returns:
        dict mapping concept -> difficulty in [0, 1]
    """
    children = defaultdict(set)
    parents = defaultdict(set)

    for h, r, t in triples:
        if r == "IS_A":
            children[t].add(h)
            parents[h].add(t)

    roots = set()
    all_nodes = set(children.keys()) | set(parents.keys())
    for concept in all_nodes:
        if concept not in parents:
            roots.add(concept)

    if not roots:
        roots = all_nodes

    depths: Dict[str, int] = {}
    from collections import deque
    queue = deque(roots)
    for r in roots:
        depths[r] = 0

    MAX_DEPTH = 50
    while queue:
        node = queue.popleft()
        d = depths[node]
        if d >= MAX_DEPTH:
            continue
        for child in children.get(node, []):
            if child not in depths:
                depths[child] = d + 1
                queue.append(child)

    if not depths:
        return {c: 0.5 for c in vocab_list}

    max_depth = max(depths.values(), default=1)
    max_depth = max(max_depth, 1)

    scores = {}
    for c in vocab_list:
        if c in depths:
            scores[c] = depths[c] / max_depth
        else:
            scores[c] = 0.5

    return scores


def score_triple(triple: Tuple[str, str, str],
                 concept_scores: Dict[str, float]) -> float:
    """Difficulty of a triple = max difficulty of its head and tail."""
    h, _, t = triple
    return max(concept_scores.get(h, 0.5), concept_scores.get(t, 0.5))


# ---------------------------------------------------------------------------
# 2. Progressive triple sampling
# ---------------------------------------------------------------------------

class CurriculumSampler:
    """
    Controls which triples are available each epoch.

    Early epochs: only low-difficulty triples (broad, abstract concepts).
    Later epochs: progressively include harder/more specific triples.
    By `full_data_pct` of total epochs, all triples are included.
    """

    def __init__(self, triples: List[Tuple[str, str, str]],
                 concept_scores: Dict[str, float],
                 total_epochs: int,
                 full_data_pct: float = 0.7,
                 enabled: bool = True):
        self.enabled = enabled
        self.total_epochs = total_epochs
        self.full_data_pct = full_data_pct
        self.n_triples = len(triples)

        scored = [
            (i, t, score_triple(t, concept_scores))
            for i, t in enumerate(triples)
        ]
        scored.sort(key=lambda x: x[2])

        self._sorted_orig_idx = [i for i, t, s in scored]
        self._sorted_scores = [s for i, t, s in scored]
        self.scored_triples = [(t, s) for i, t, s in scored]

    def get_epoch_triples(self, epoch: int) -> List[Tuple[str, str, str]]:
        if not self.enabled:
            return [t for t, _ in self.scored_triples]

        threshold = min(1.0, (epoch + 1) / max(self.total_epochs * self.full_data_pct, 1))
        result = [t for t, s in self.scored_triples if s <= threshold]

        if len(result) < max(100, len(self.scored_triples) // 10):
            result = [t for t, _ in self.scored_triples[:max(100, len(self.scored_triples) // 10)]]

        return result

    def get_epoch_indices(self, epoch: int) -> torch.Tensor:
        """Return original triple indices for this epoch as a LongTensor."""
        if not self.enabled:
            return torch.arange(self.n_triples, dtype=torch.long)

        threshold = min(1.0, (epoch + 1) / max(self.total_epochs * self.full_data_pct, 1))
        selected = [
            self._sorted_orig_idx[i]
            for i, s in enumerate(self._sorted_scores)
            if s <= threshold
        ]

        min_count = max(100, self.n_triples // 10)
        if len(selected) < min_count:
            selected = self._sorted_orig_idx[:min_count]

        return torch.tensor(selected, dtype=torch.long)

    def get_progress(self, epoch: int) -> float:
        """Returns curriculum progress in [0, 1]."""
        return min(1.0, (epoch + 1) / max(self.total_epochs * self.full_data_pct, 1))


# ---------------------------------------------------------------------------
# 3. Loss weight scheduling
# ---------------------------------------------------------------------------

class LossWeightScheduler:
    """
    Gradually introduces harder losses over training.

    L_rel: always active (foundational KG structure)
    L_comp: ramps in over first 30% of training
    L_cl: ramps in over first 50%
    L_ent: starts at 20%, full at 60%
    L_hier: starts at 30%, full at 70%

    Base weights from config are multiplied by the schedule factor.
    """

    def __init__(self, total_epochs: int, enabled: bool = True):
        self.total_epochs = max(total_epochs, 1)
        self.enabled = enabled

    def get_weights(self, epoch: int, base_weights: Dict[str, float]) -> Dict[str, float]:
        if not self.enabled:
            return dict(base_weights)

        progress = epoch / self.total_epochs

        schedule = {
            "L_rel": 1.0,
            "L_comp": min(1.0, progress / 0.3) if progress > 0 else 0.0,
            "L_cl": min(1.0, progress / 0.5) if progress > 0 else 0.0,
            "L_ent": min(1.0, max(0, (progress - 0.2) / 0.4)),
            "L_hier": min(1.0, max(0, (progress - 0.3) / 0.4)),
        }

        return {
            k: base_weights.get(k, 1.0) * schedule.get(k, 1.0)
            for k in base_weights
        }


# ---------------------------------------------------------------------------
# 4. Hard negative mining
# ---------------------------------------------------------------------------

def sample_negatives(
    batch_size: int,
    vocab_list: List[str],
    epoch: int,
    total_epochs: int,
    all_z: Optional[torch.Tensor] = None,
    z_pred: Optional[torch.Tensor] = None,
    vocab2idx: Optional[Dict[str, int]] = None,
    exclude_idx: Optional[List[int]] = None,
    hard_neg_max_ratio: float = 0.8,
) -> List[str]:
    """
    Sample negative tails with increasing hard-negative ratio.

    Easy negatives: random vocab samples.
    Hard negatives: nearest non-target entities in current embedding space.
    """
    hard_ratio = min(hard_neg_max_ratio, epoch / max(total_epochs * 0.6, 1))
    n_hard = int(batch_size * hard_ratio)
    n_easy = batch_size - n_hard

    easy_negs = random.choices(vocab_list, k=n_easy)

    if n_hard > 0 and all_z is not None and z_pred is not None and vocab2idx is not None:
        with torch.no_grad():
            dists = torch.cdist(z_pred[:n_hard].float(), all_z.float(), p=2)

            if exclude_idx is not None:
                for i, eidx in enumerate(exclude_idx[:n_hard]):
                    if eidx is not None:
                        dists[i, eidx] = float("inf")

            hard_indices = dists.argmin(dim=-1).tolist()
            hard_negs = [vocab_list[idx] for idx in hard_indices]
    else:
        hard_negs = random.choices(vocab_list, k=n_hard)

    return easy_negs + hard_negs


def sample_negatives_gpu(
    batch_size: int,
    n_vocab: int,
    epoch: int,
    total_epochs: int,
    device: torch.device,
    all_z: Optional[torch.Tensor] = None,
    z_pred: Optional[torch.Tensor] = None,
    t_idx: Optional[torch.Tensor] = None,
    hard_neg_max_ratio: float = 0.8,
) -> torch.Tensor:
    """
    GPU-native negative sampling — returns vocab index tensor directly.

    No string operations, no CPU round-trips.
    """
    hard_ratio = min(hard_neg_max_ratio, epoch / max(total_epochs * 0.6, 1))
    n_hard = int(batch_size * hard_ratio)
    n_easy = batch_size - n_hard

    neg_idx = torch.randint(0, n_vocab, (batch_size,), device=device)

    if n_hard > 0 and all_z is not None and z_pred is not None:
        with torch.no_grad():
            dists = torch.cdist(z_pred[:n_hard].float(), all_z.float(), p=2)
            if t_idx is not None:
                dists[torch.arange(n_hard, device=device), t_idx[:n_hard]] = float("inf")
            neg_idx[:n_hard] = dists.argmin(dim=-1)

    return neg_idx

# ----- data.py -----
import re
import random
from collections import defaultdict, Counter
from typing import List, Tuple, Dict

import torch
from torch.utils.data import Dataset, DataLoader



# ---------------------------------------------------------------------------
# ConceptNet relation mapping
# ---------------------------------------------------------------------------

_CN_MAP = {
    "/r/IsA": "IS_A", "IsA": "IS_A", "is_a": "IS_A",
    "/r/Causes": "CAUSES", "Causes": "CAUSES", "causes": "CAUSES",
    "/r/PartOf": "PART_OF", "PartOf": "PART_OF", "part_of": "PART_OF",
    "/r/SimilarTo": "SIMILAR_TO", "SimilarTo": "SIMILAR_TO", "similar_to": "SIMILAR_TO",
    "/r/Antonym": "ANTONYM", "Antonym": "ANTONYM", "antonym": "ANTONYM",
    "/r/CapableOf": "CAPABLE_OF", "CapableOf": "CAPABLE_OF", "capable_of": "CAPABLE_OF",
}


def _norm_concept(s: str):
    s = str(s).strip()
    if "/c/en/" in s:
        s = s.split("/c/en/")[-1]
        s = s.split("/")[0]
    elif "/" in s:
        s = s.split("/")[-1]
    s = re.sub(r"[^a-z ]", "", s.replace("_", " ").lower()).strip()
    return s if len(s) > 1 else None


# ---------------------------------------------------------------------------
# Hardcoded fallback triples
# ---------------------------------------------------------------------------

_FALLBACK_TRIPLES = [
    ("dog", "IS_A", "animal"), ("cat", "IS_A", "animal"), ("wolf", "IS_A", "animal"),
    ("eagle", "IS_A", "bird"), ("robin", "IS_A", "bird"), ("penguin", "IS_A", "bird"),
    ("salmon", "IS_A", "fish"), ("shark", "IS_A", "fish"),
    ("rose", "IS_A", "flower"), ("tulip", "IS_A", "flower"), ("daisy", "IS_A", "flower"),
    ("oak", "IS_A", "tree"), ("pine", "IS_A", "tree"), ("maple", "IS_A", "tree"),
    ("hammer", "IS_A", "tool"), ("wrench", "IS_A", "tool"), ("saw", "IS_A", "tool"),
    ("violin", "IS_A", "instrument"), ("piano", "IS_A", "instrument"),
    ("fire", "CAUSES", "smoke"), ("fire", "CAUSES", "heat"),
    ("rain", "CAUSES", "flooding"), ("exercise", "CAUSES", "fatigue"),
    ("wheel", "PART_OF", "car"), ("engine", "PART_OF", "car"),
    ("leaf", "PART_OF", "tree"), ("branch", "PART_OF", "tree"),
    ("happy", "SIMILAR_TO", "joyful"), ("sad", "SIMILAR_TO", "unhappy"),
    ("fast", "SIMILAR_TO", "quick"), ("big", "SIMILAR_TO", "large"),
    ("hot", "ANTONYM", "cold"), ("fast", "ANTONYM", "slow"),
    ("love", "ANTONYM", "hate"), ("light", "ANTONYM", "dark"),
    ("dog", "CAPABLE_OF", "barking"), ("bird", "CAPABLE_OF", "flying"),
    ("fish", "CAPABLE_OF", "swimming"), ("human", "CAPABLE_OF", "thinking"),
    ("lion", "IS_A", "animal"), ("bear", "IS_A", "animal"), ("horse", "IS_A", "animal"),
    ("sparrow", "IS_A", "bird"), ("hawk", "IS_A", "bird"), ("owl", "IS_A", "bird"),
    ("lily", "IS_A", "flower"), ("orchid", "IS_A", "flower"),
    ("birch", "IS_A", "tree"), ("willow", "IS_A", "tree"),
    ("drum", "IS_A", "instrument"), ("flute", "IS_A", "instrument"),
    ("chair", "IS_A", "furniture"), ("table", "IS_A", "furniture"),
    ("soccer", "IS_A", "sport"), ("tennis", "IS_A", "sport"),
    ("lightning", "CAUSES", "fire"), ("virus", "CAUSES", "disease"),
    ("roof", "PART_OF", "house"), ("keyboard", "PART_OF", "computer"),
    ("angry", "SIMILAR_TO", "furious"), ("tiny", "SIMILAR_TO", "small"),
    ("rich", "ANTONYM", "poor"), ("strong", "ANTONYM", "weak"),
    ("horse", "CAPABLE_OF", "galloping"), ("eagle", "CAPABLE_OF", "soaring"),
]


def _augment_triples(triples):
    augmented = list(triples)
    for h, r, t in triples:
        if r in ("SIMILAR_TO", "ANTONYM"):
            augmented.append((t, r, h))
    isa_groups = defaultdict(list)
    for h, r, t in triples:
        if r == "IS_A":
            isa_groups[t].append(h)
    for parent, children in isa_groups.items():
        for i, c1 in enumerate(children):
            for c2 in children[i + 1:]:
                augmented.append((c1, "SIMILAR_TO", c2))
                augmented.append((c2, "SIMILAR_TO", c1))
    return augmented


# ---------------------------------------------------------------------------
# ConceptNet loading
# ---------------------------------------------------------------------------

def _load_conceptnet_s3(max_triples: int = 100_000):
    import urllib.request
    import gzip
    import io as _io

    url = "https://s3.amazonaws.com/conceptnet/downloads/2019/edges/conceptnet-assertions-5.7.0.csv.gz"
    print("  Streaming ConceptNet 5.7 from S3 ...")
    resp = urllib.request.urlopen(url, timeout=30)
    reader = _io.TextIOWrapper(gzip.GzipFile(fileobj=resp), encoding="utf-8")
    triples = []
    scanned = 0
    for line in reader:
        scanned += 1
        parts = line.strip().split("\t")
        if len(parts) < 5:
            continue
        rel, head, tail = parts[1], parts[2], parts[3]
        if not head.startswith("/c/en/") or not tail.startswith("/c/en/"):
            continue
        r = _CN_MAP.get(rel)
        if not r:
            continue
        h = _norm_concept(head)
        t = _norm_concept(tail)
        if h and t and h != t:
            triples.append((h, r, t))
        if len(triples) >= max_triples:
            break
        if scanned % 2_000_000 == 0:
            print(f"    ...{scanned // 1_000_000}M rows, {len(triples):,} triples so far")
    reader.close()
    return triples


def load_conceptnet(max_triples: int = 100_000) -> List[Tuple[str, str, str]]:
    try:
        triples = _load_conceptnet_s3(max_triples)
        if triples:
            rc = Counter(r for _, r, _ in triples)
            print(f"  OK  {len(triples):,} triples from S3 ({dict(rc)})")
            return triples
    except Exception as e:
        print(f"  FAIL  S3 streaming: {e}")

    try:
        from datasets import load_dataset
        print("  Trying conceptnet5/conceptnet5 (HuggingFace streaming)...")
        ds = load_dataset("conceptnet5/conceptnet5", split="train", streaming=True)
        triples = []
        for row in ds:
            if row.get("lang") != "en":
                continue
            r = _CN_MAP.get(row.get("rel"))
            if not r:
                continue
            h = _norm_concept(row["arg1"])
            t = _norm_concept(row["arg2"])
            if h and t and h != t:
                triples.append((h, r, t))
            if len(triples) >= max_triples:
                break
        if triples:
            print(f"  OK  {len(triples):,} triples from HuggingFace")
            return triples
    except Exception as e:
        print(f"  FAIL  HuggingFace: {e}")

    augmented = _augment_triples(_FALLBACK_TRIPLES)
    print(f"  Using hardcoded fallback: {len(_FALLBACK_TRIPLES)} base + "
          f"{len(augmented) - len(_FALLBACK_TRIPLES)} augmented = {len(augmented)} triples")
    return augmented


# ---------------------------------------------------------------------------
# Cross-lingual
# ---------------------------------------------------------------------------

def load_crosslingual(max_per_lang: int = 20_000) -> List[Tuple[str, str]]:
    from datasets import load_dataset
    pairs = []
    for lang in ["en-fr", "en-de", "en-es"]:
        try:
            print(f"  Trying tatoeba {lang}...")
            ds = load_dataset("sentence-transformers/parallel-sentences-tatoeba",
                              lang, split="train")
            lp = [(r["english"].strip(), r["non_english"].strip())
                  for r in ds if r["english"].strip() and r["non_english"].strip()]
            pairs.extend(lp[:max_per_lang])
            print(f"  OK  {min(len(lp), max_per_lang)} pairs from tatoeba {lang}")
        except Exception as e:
            print(f"  FAIL  tatoeba {lang}: {e}")

    if not pairs:
        print("  Using synthetic fallback (EN-FR/DE/ES)")
        pairs = [
            ("A dog is a domesticated animal.", "Un chien est un animal domestiqué."),
            ("Fire causes smoke.", "Le feu cause de la fumée."),
            ("A dog is a domesticated animal.", "Ein Hund ist ein domestiziertes Tier."),
            ("Fire causes smoke.", "Feuer verursacht Rauch."),
            ("A dog is a domesticated animal.", "Un perro es un animal domesticado."),
            ("Fire causes smoke.", "El fuego causa humo."),
        ] * 20
    return pairs


# ---------------------------------------------------------------------------
# SNLI
# ---------------------------------------------------------------------------

def load_snli(max_pairs: int = 20_000) -> List[Tuple[str, str, int]]:
    try:
        from datasets import load_dataset
        ds = load_dataset("snli", split=f"train[:{max_pairs}]")
        pairs = [(r["premise"], r["hypothesis"], r["label"])
                 for r in ds if r["label"] in (0, 1, 2)]
        print(f"  OK  {len(pairs)} SNLI pairs")
        return pairs
    except Exception as e:
        print(f"  FAIL  SNLI: {e}")

    print("  Building SNLI-style fallback from hardcoded ConceptNet...")
    pairs = []
    for h, r, t in _FALLBACK_TRIPLES:
        if r == "IS_A":
            pairs.append((f"A {h} is an example.", f"A {t} is an example.", 0))
            pairs.append((f"A {h} is an example.", "Something unrelated happened.", 1))
        elif r == "ANTONYM":
            pairs.append((f"{h} is the topic.", f"{t} is the topic.", 2))
    print(f"  OK  {len(pairs)} fallback SNLI-style pairs")
    return pairs


# ---------------------------------------------------------------------------
# CIFAR-100 constants
# ---------------------------------------------------------------------------

CIFAR100_COARSE = [
    "aquatic mammals", "fish", "flowers", "food containers", "fruit and vegetables",
    "household electrical devices", "household furniture", "insects", "large carnivores",
    "large man-made outdoor things", "large natural outdoor scenes",
    "large omnivores and herbivores", "medium-sized mammals", "non-insect invertebrates",
    "people", "reptiles", "small mammals", "trees", "vehicles 1", "vehicles 2",
]

CIFAR100_FINE = [
    "apple", "aquarium fish", "baby", "bear", "beaver", "bed", "bee", "beetle",
    "bicycle", "bottle", "bowl", "boy", "bridge", "bus", "butterfly", "camel",
    "can", "castle", "caterpillar", "cattle", "chair", "chimpanzee", "clock",
    "cloud", "cockroach", "couch", "crab", "crocodile", "cup", "dinosaur",
    "dolphin", "elephant", "flatfish", "forest", "fox", "girl", "hamster",
    "house", "kangaroo", "keyboard", "lamp", "lawn mower", "leopard", "lion",
    "lizard", "lobster", "man", "maple tree", "motorcycle", "mountain", "mouse",
    "mushroom", "oak tree", "orange", "orchid", "otter", "palm tree", "pear",
    "pickup truck", "pine tree", "plain", "plate", "poppy", "porcupine",
    "possum", "rabbit", "raccoon", "ray", "road", "rocket", "rose", "sea",
    "seal", "shark", "shrew", "skunk", "skyscraper", "snail", "snake",
    "spider", "squirrel", "streetcar", "sunflower", "sweet pepper", "table",
    "tank", "telephone", "television", "tiger", "tractor", "train", "trout",
    "tulip", "turtle", "wardrobe", "whale", "willow tree", "wolf", "woman", "worm",
]

CIFAR100_FINE2COARSE = {
    0: 4, 1: 1, 2: 14, 3: 8, 4: 0, 5: 6, 6: 7, 7: 7, 8: 18, 9: 3, 10: 3, 11: 14,
    12: 9, 13: 18, 14: 7, 15: 11, 16: 3, 17: 9, 18: 7, 19: 11, 20: 6, 21: 11, 22: 5,
    23: 10, 24: 7, 25: 6, 26: 13, 27: 15, 28: 3, 29: 15, 30: 0, 31: 11, 32: 1, 33: 10,
    34: 12, 35: 14, 36: 16, 37: 9, 38: 11, 39: 5, 40: 5, 41: 19, 42: 8, 43: 8, 44: 15,
    45: 13, 46: 14, 47: 17, 48: 18, 49: 10, 50: 16, 51: 4, 52: 17, 53: 4, 54: 2, 55: 0,
    56: 17, 57: 4, 58: 18, 59: 17, 60: 10, 61: 3, 62: 2, 63: 12, 64: 12, 65: 16, 66: 12,
    67: 1, 68: 9, 69: 19, 70: 2, 71: 10, 72: 0, 73: 1, 74: 16, 75: 12, 76: 9, 77: 13,
    78: 15, 79: 13, 80: 16, 81: 19, 82: 2, 83: 4, 84: 6, 85: 19, 86: 5, 87: 5, 88: 8,
    89: 19, 90: 18, 91: 1, 92: 2, 93: 15, 94: 6, 95: 0, 96: 17, 97: 8, 98: 14, 99: 13,
}


def load_cifar100(data_root: str = "./data"):
    import torchvision
    import torchvision.transforms as T

    transform = T.Compose([
        T.Resize(224), T.CenterCrop(224), T.ToTensor(),
        T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
    ])
    train = torchvision.datasets.CIFAR100(
        root=data_root, train=True, download=True, transform=transform)
    test = torchvision.datasets.CIFAR100(
        root=data_root, train=False, download=True, transform=transform)
    return train, test


# ---------------------------------------------------------------------------
# KG Dataset for DataLoader
# ---------------------------------------------------------------------------

class KGDataset(Dataset):
    def __init__(self, triples: List[Tuple[str, str, str]]):
        self.triples = triples

    def __len__(self):
        return len(self.triples)

    def __getitem__(self, idx):
        h, r, t = self.triples[idx]
        return h, REL2IDX[r], t


def kg_collate_fn(batch):
    heads, rels, tails = zip(*batch)
    return list(heads), torch.tensor(rels, dtype=torch.long), list(tails)

# ----- training.py -----
import os
import time
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from typing import Dict, List, Optional

import geoopt



def _precache_text_embeddings(
    encoder: ConceptEncoder,
    texts: List[str],
    batch_size: int = 512,
    device: torch.device = torch.device("cpu"),
) -> torch.Tensor:
    """Pre-encode texts through the frozen backbone (run once, reuse)."""
    all_embs = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Pre-caching text", leave=False):
        emb = encoder.encode_backbone(texts[i:i + batch_size])
        all_embs.append(emb.cpu())
    return torch.cat(all_embs, dim=0).to(device)


def _precache_clip_embeddings(
    vis_encoder: VisionEncoder,
    dataloader: DataLoader,
    device: torch.device = torch.device("cpu"),
    max_images: int = 50000,
) -> tuple:
    """Pre-encode CIFAR images through the frozen CLIP backbone."""
    all_embs, all_labels = [], []
    n = 0
    for imgs, labels in tqdm(dataloader, desc="Pre-caching CLIP", leave=False):
        emb = vis_encoder.encode_images(imgs.to(device))
        all_embs.append(emb.cpu())
        all_labels.append(labels)
        n += imgs.shape[0]
        if n >= max_images:
            break
    return torch.cat(all_embs, dim=0).to(device), torch.cat(all_labels, dim=0).to(device)


# ---------------------------------------------------------------------------
# Phase 1: Text-only training
# ---------------------------------------------------------------------------

def train_phase1(
    cfg: USMConfig,
    manifold: LearnablePoincareBall,
    encoder: ConceptEncoder,
    comp_op: CompositionalOperator,
    rel_maps: RelationMaps,
    triples: List,
    cl_pairs: List,
    snli_pairs: List,
    vocab_list: List[str],
    vocab_bb: torch.Tensor,
    concept2bb: Dict[str, int],
    cl_bb_src: Optional[torch.Tensor] = None,
    cl_bb_tgt: Optional[torch.Tensor] = None,
    snli_bb_a: Optional[torch.Tensor] = None,
    snli_bb_b: Optional[torch.Tensor] = None,
    snli_labels_t: Optional[torch.Tensor] = None,
) -> Dict[str, List]:
    """
    Phase 1 — GPU-native training loop.

    All triple data is pre-converted to integer index tensors on GPU.
    No string operations, dict lookups, or CPU-side DataLoader in the hot loop.
    """
    device = cfg.device
    history = {
        "loss_total": [], "loss_rel": [], "loss_comp": [],
        "loss_cl": [], "loss_ent": [], "loss_hier": [],
        "curvature": [], "curvature_per_rel": [], "lr": [], "curriculum_pct": [],
    }

    # ---- Pre-convert ALL triples to GPU index tensors (once) ----
    all_h_idx = torch.tensor([concept2bb[h] for h, r, t in triples], device=device)
    all_r_idx = torch.tensor([REL2IDX[r] for h, r, t in triples], device=device)
    all_t_idx = torch.tensor([concept2bb[t] for h, r, t in triples], device=device)
    n_vocab = len(vocab_list)

    difficulty_scores = compute_difficulty_scores(triples, vocab_list)
    curriculum = CurriculumSampler(
        triples, difficulty_scores, cfg.n_epochs_p1,
        full_data_pct=cfg.curriculum_full_data_pct,
        enabled=cfg.curriculum_enabled,
    )
    loss_scheduler = LossWeightScheduler(cfg.n_epochs_p1, enabled=cfg.curriculum_enabled)
    burnin = BurninScheduler(cfg.burnin_epochs, cfg.transition_epochs)

    n_cl = cl_bb_src.shape[0] if cl_bb_src is not None else 0
    n_snli = snli_bb_a.shape[0] if snli_bb_a is not None else 0

    # Curvature is scheduled, not learned — prevents optimizer collapse
    manifold._c_params.requires_grad_(False)
    c_targets = torch.tensor(
        cfg.c_targets[:N_RELATIONS] if hasattr(cfg, 'c_targets') else [cfg.c_init] * N_RELATIONS,
        device=device, dtype=torch.float32,
    )
    c_ws = getattr(cfg, 'c_warmup_start', 10)
    c_we = getattr(cfg, 'c_warmup_end', 40)

    manifold_ids = {id(p) for p in manifold.parameters()}
    model_params = []
    seen = set(manifold_ids)
    for p in (
        list(encoder.proj.parameters())
        + ([encoder.mu0] if hasattr(encoder, "mu0") else [])
        + list(comp_op.parameters())
        + list(rel_maps.parameters())
    ):
        pid = id(p)
        if pid not in seen:
            seen.add(pid)
            model_params.append(p)

    optimizer = geoopt.optim.RiemannianAdam(
        [{"params": model_params, "lr": cfg.lr_p1}],
        weight_decay=cfg.weight_decay,
    )

    base_weights = {
        "L_rel": cfg.w_rel, "L_comp": cfg.w_comp,
        "L_cl": cfg.w_cl, "L_ent": cfg.w_ent, "L_hier": cfg.w_hier,
    }

    ISA_IDX = REL2IDX.get("IS_A", 0)
    PARTOF_IDX = REL2IDX.get("PART_OF", 2)

    e0_idx = curriculum.get_epoch_indices(0)
    e0_batches = len(e0_idx) // cfg.batch_size
    tgt_str = " ".join(f"{v:.4f}" for v in c_targets.tolist())
    print(
        f'Phase 1 starting: {cfg.n_epochs_p1} epochs | {len(triples):,} triples | '
        f'batch={cfg.batch_size} | d={cfg.d} | device={device}\n'
        f'  Curvature targets: [{tgt_str}]\n'
        f'  Relations:         {" ".join(RELATIONS)}\n'
        f'  Warmup: E{c_ws}→E{c_we} (linear ramp per relation)\n'
        f'  Epoch 0: {len(e0_idx):,} triples (~{e0_batches} batches)\n'
        f'  GPU-native loop + cosine LR schedule',
        flush=True,
    )

    for epoch in range(cfg.n_epochs_p1):
        t0 = time.time()
        encoder.train()
        comp_op.train()
        rel_maps.train()

        # Scheduled curvature warmup: linear ramp from c_init to c_targets
        with torch.no_grad():
            if epoch < c_ws:
                progress = 0.0
            elif epoch >= c_we:
                progress = 1.0
            else:
                progress = (epoch - c_ws) / max(c_we - c_ws, 1)
            manifold._c_params.copy_(cfg.c_init + (c_targets - cfg.c_init) * progress)

        # Cosine LR schedule
        cos_lr = cfg.lr_p1 * 0.5 * (1 + math.cos(math.pi * epoch / cfg.n_epochs_p1))
        optimizer.param_groups[0]["lr"] = cos_lr

        c_scale = burnin.get_curvature_scale(epoch)

        epoch_idx = curriculum.get_epoch_indices(epoch).to(device)
        n_epoch = len(epoch_idx)
        weights = loss_scheduler.get_weights(epoch, base_weights)
        cpct = curriculum.get_progress(epoch)

        # Shuffle this epoch's triples on GPU
        perm = torch.randperm(n_epoch, device=device)
        ep_h = all_h_idx[epoch_idx[perm]]
        ep_r = all_r_idx[epoch_idx[perm]]
        ep_t = all_t_idx[epoch_idx[perm]]

        c_base_detached = manifold.c.detach()
        all_z = (
            encoder.project(vocab_bb, c=c_base_detached).detach()
            if epoch > 5 else None
        )

        epoch_loss = 0.0
        epoch_losses = {k: 0.0 for k in history if k.startswith("loss_")}
        n_batches = 0
        BS = cfg.batch_size
        n_total_batches = n_epoch // BS

        batch_iter = tqdm(
            range(0, n_epoch - BS + 1, BS),
            desc=f'P1 E{epoch:02d}',
            total=n_total_batches,
            leave=(epoch == cfg.n_epochs_p1 - 1),
        )
        for start in batch_iter:
            c_base = manifold.effective_c(c_scale)

            h_idx = ep_h[start:start + BS]
            t_idx = ep_t[start:start + BS]
            rels = ep_r[start:start + BS]

            z_h = encoder.project(vocab_bb[h_idx], c=c_base)
            z_t = encoder.project(vocab_bb[t_idx], c=c_base)

            neg_idx = sample_negatives_gpu(
                BS, n_vocab, epoch, cfg.n_epochs_p1, device,
                all_z=all_z,
                z_pred=None,
                t_idx=t_idx,
                hard_neg_max_ratio=cfg.hard_neg_max_ratio,
            )
            z_neg = encoder.project(vocab_bb[neg_idx], c=c_base)

            # --- Per-relation loss: each relation uses its own curvature ---
            L_rel_val = z_h.new_zeros(())
            L_hier_val = z_h.new_zeros(())
            rel_count = 0
            hier_count = 0
            for r in range(N_RELATIONS):
                mask = (rels == r)
                n_r = mask.sum().item()
                if n_r < 2:
                    continue
                c_r = manifold.effective_c_rel(r, c_scale)
                z_pred_r = rel_maps(z_h[mask], rels[mask], c=c_r)
                L_r = loss_rel(z_pred_r, z_t[mask], z_neg[mask], c_r, margin=cfg.margin_rel)
                L_rel_val = L_rel_val + L_r * n_r
                rel_count += n_r

                if r in (ISA_IDX, PARTOF_IDX) and c_r.item() > EPS and weights.get("L_hier", 0) > 0:
                    depth_h_r = dist0(z_h[mask], c_r)
                    depth_t_r = dist0(z_t[mask], c_r)
                    L_hier_val = L_hier_val + F.relu(depth_t_r - depth_h_r + cfg.margin_hier).mean() * n_r
                    hier_count += n_r

            if rel_count > 0:
                L_rel_val = L_rel_val / rel_count
            if hier_count > 0:
                L_hier_val = L_hier_val / hier_count

            z_comp = comp_op(z_h, rels, z_t, c=c_base)
            if c_base.item() > EPS:
                L_comp_val = logmap0(z_comp, c_base).pow(2).mean()
            else:
                L_comp_val = z_comp.pow(2).mean()

            L_cl_val = z_h.new_zeros(())
            if n_cl > 0 and weights.get("L_cl", 0) > 0:
                n_cl_batch = min(BS // 2, n_cl)
                idx_cl = torch.randint(0, n_cl, (n_cl_batch,), device=device)
                z_src = encoder.project(cl_bb_src[idx_cl], c=c_base)
                z_tgt = encoder.project(cl_bb_tgt[idx_cl], c=c_base)
                L_cl_val = loss_cl(z_src, z_tgt, c_base)

            L_ent_val = z_h.new_zeros(())
            if n_snli > 0 and weights.get("L_ent", 0) > 0:
                n_ent_batch = min(BS // 2, n_snli)
                idx_sn = torch.randint(0, n_snli, (n_ent_batch,), device=device)
                z_a_ent = encoder.project(snli_bb_a[idx_sn], c=c_base)
                z_b_ent = encoder.project(snli_bb_b[idx_sn], c=c_base)
                L_ent_val = loss_entailment(
                    z_a_ent, z_b_ent,
                    snli_labels_t[idx_sn],
                    c_base, delta_c=cfg.delta_contradiction,
                )

            total = (
                weights["L_rel"] * L_rel_val
                + weights["L_comp"] * L_comp_val
                + weights.get("L_cl", 0) * L_cl_val
                + weights.get("L_ent", 0) * L_ent_val
                + weights.get("L_hier", 0) * L_hier_val
            )

            total.backward()

            if c_base.item() > EPS and cfg.use_riemannian_clipping:
                manifold_aware = [p for p in model_params if p.grad is not None]
                scale_riemannian_grads(manifold_aware, c_base)
                clip_riemannian_grad_norm(manifold_aware, cfg.grad_clip_norm, c_base)
            else:
                torch.nn.utils.clip_grad_norm_(model_params, cfg.grad_clip_norm)

            optimizer.step()
            optimizer.zero_grad()

            epoch_loss += total.item()
            epoch_losses["loss_rel"] += L_rel_val.item()
            epoch_losses["loss_comp"] += L_comp_val.item()
            epoch_losses["loss_cl"] += L_cl_val.item()
            epoch_losses["loss_ent"] += L_ent_val.item()
            epoch_losses["loss_hier"] += L_hier_val.item()
            n_batches += 1

            if n_batches == 1 or n_batches % 25 == 0:
                batch_iter.set_postfix(
                    loss=f'{total.item():.3f}',
                    c_base=f'{c_base.item():.3f}',
                )

        nb = max(n_batches, 1)
        history["loss_total"].append(epoch_loss / nb)
        history["loss_rel"].append(epoch_losses["loss_rel"] / nb)
        history["loss_comp"].append(epoch_losses["loss_comp"] / nb)
        history["loss_cl"].append(epoch_losses["loss_cl"] / nb)
        history["loss_ent"].append(epoch_losses["loss_ent"] / nb)
        history["loss_hier"].append(epoch_losses["loss_hier"] / nb)
        c_per_rel = [manifold.c_rel(r).item() for r in range(N_RELATIONS)]
        history["curvature"].append(max(c_per_rel))
        history["curvature_per_rel"].append(c_per_rel)
        history["lr"].append(cos_lr)
        history["curriculum_pct"].append(cpct)

        dt = time.time() - t0
        c_rel_str = " ".join(f"{v:.3f}" for v in c_per_rel)
        print(
            f"[P1 E{epoch:02d}] "
            f"loss={epoch_loss / nb:.4f}  "
            f"c=[{c_rel_str}]  "
            f"lr={cos_lr:.1e}  "
            f"data={n_epoch:,}/{len(triples):,}  "
            f"curriculum={cpct:.0%}  "
            f"({dt:.1f}s)",
            flush=True,
        )

        if cfg.ckpt_dir and (epoch + 1) % cfg.ckpt_every == 0:
            os.makedirs(cfg.ckpt_dir, exist_ok=True)
            torch.save({
                "epoch": epoch,
                "encoder": encoder.state_dict(),
                "comp_op": comp_op.state_dict(),
                "rel_maps": rel_maps.state_dict(),
                "manifold": manifold.state_dict(),
                "optimizer": optimizer.state_dict(),
                "config": cfg,
            }, os.path.join(cfg.ckpt_dir, f"p1_epoch_{epoch:03d}.pt"))

    return history


# ---------------------------------------------------------------------------
# Phase 2: Multimodal training
# ---------------------------------------------------------------------------

def train_phase2(
    cfg: USMConfig,
    manifold: LearnablePoincareBall,
    encoder: ConceptEncoder,
    vis_encoder: VisionEncoder,
    precomp_clip: torch.Tensor,
    precomp_labels: torch.Tensor,
    fine_bb: torch.Tensor,
    coarse_bb: torch.Tensor,
    fine2coarse_tensor: torch.Tensor,
) -> Dict[str, List]:
    """
    Phase 2: align vision with frozen text geometry + hierarchy.

    Curvature continues from Phase 1 (already ramped up).
    Gradient control remains active.
    """
    device = cfg.device
    history = {
        "loss_total": [], "loss_xmodal": [], "loss_hier": [],
        "curvature": [],
    }

    manifold_ids = {id(p) for p in manifold.parameters()}
    manifold_params = [p for p in manifold.parameters() if p.requires_grad] if cfg.learnable_curvature else []

    seen = set(id(p) for p in manifold_params) | manifold_ids
    vis_params = []
    for p in list(vis_encoder.proj.parameters()) + ([vis_encoder.mu0] if hasattr(vis_encoder, "mu0") else []):
        if id(p) not in seen:
            seen.add(id(p))
            vis_params.append(p)

    text_params = []
    for p in list(encoder.proj.parameters()) + ([encoder.mu0] if hasattr(encoder, "mu0") else []):
        if id(p) not in seen:
            seen.add(id(p))
            text_params.append(p)

    param_groups = [
        {"params": vis_params, "lr": cfg.lr_p2_vis},
        {"params": text_params, "lr": cfg.lr_p2_text},
    ]
    if manifold_params:
        param_groups.append({"params": manifold_params, "lr": cfg.c_lr * 0.1})

    optimizer = geoopt.optim.RiemannianAdam(param_groups, weight_decay=cfg.weight_decay)

    N = precomp_clip.shape[0]
    p2_batches = (N + cfg.p2_batch - 1) // cfg.p2_batch
    print(
        f'Phase 2 starting: {cfg.n_epochs_p2} epochs | {N:,} images | '
        f'batch={cfg.p2_batch} (~{p2_batches} batches/epoch) | device={device}',
        flush=True,
    )

    for epoch in range(cfg.n_epochs_p2):
        t0 = time.time()
        vis_encoder.train()
        encoder.train()

        perm = torch.randperm(N, device=device)
        epoch_loss = 0.0
        n_batches = 0

        batch_starts = range(0, N, cfg.p2_batch)
        batch_iter = tqdm(batch_starts, desc=f'P2 E{epoch:02d}', leave=(epoch == cfg.n_epochs_p2 - 1))
        for start in batch_iter:
            c_eff = manifold.c

            idx = perm[start:start + cfg.p2_batch]
            clip_emb = precomp_clip[idx]
            labels = precomp_labels[idx]

            z_img = vis_encoder.project(clip_emb, c=c_eff)

            fine_labels = labels
            coarse_labels = fine2coarse_tensor[labels]
            z_fine_text = encoder.project(fine_bb[fine_labels], c=c_eff)
            z_coarse_text = encoder.project(coarse_bb[coarse_labels], c=c_eff)

            L_xm = loss_crossmodal(z_fine_text, z_img, c_eff)
            L_hier = loss_hierarchy(z_fine_text, z_coarse_text, c_eff, margin=cfg.margin_hier)

            total = L_xm + cfg.w_hier * L_hier
            total.backward()

            all_params = vis_params + text_params
            if c_eff.item() > EPS and cfg.use_riemannian_clipping:
                manifold_aware = [p for p in all_params if p.grad is not None]
                scale_riemannian_grads(manifold_aware, c_eff)
                clip_riemannian_grad_norm(manifold_aware, cfg.grad_clip_norm, c_eff)
            else:
                torch.nn.utils.clip_grad_norm_(all_params, cfg.grad_clip_norm)

            optimizer.step()
            optimizer.zero_grad()

            epoch_loss += total.item()
            n_batches += 1

            if n_batches == 1 or n_batches % 10 == 0:
                batch_iter.set_postfix(loss=f'{total.item():.3f}')

        nb = max(n_batches, 1)
        history["loss_total"].append(epoch_loss / nb)
        history["loss_xmodal"].append(0.0)
        history["loss_hier"].append(0.0)
        with torch.no_grad():
            c_val = manifold.c.item()
        history["curvature"].append(c_val)

        dt = time.time() - t0
        print(
            f"[P2 E{epoch:02d}] "
            f"loss={epoch_loss / nb:.4f}  "
            f"c={c_val:.4f}  "
            f"({dt:.1f}s)",
            flush=True,
        )

        if cfg.ckpt_dir and (epoch + 1) % cfg.ckpt_every == 0:
            os.makedirs(cfg.ckpt_dir, exist_ok=True)
            torch.save({
                "epoch": epoch,
                "encoder": encoder.state_dict(),
                "vis_encoder": vis_encoder.state_dict(),
                "manifold": manifold.state_dict(),
                "optimizer": optimizer.state_dict(),
                "config": cfg,
            }, os.path.join(cfg.ckpt_dir, f"p2_epoch_{epoch:03d}.pt"))

    return history

# ----- evaluation.py -----
import numpy as np
import torch
from tqdm.auto import tqdm
from typing import List, Tuple, Dict, Optional



def evaluate_link_prediction(
    encoder,
    rel_maps,
    test_triples: List[Tuple[str, str, str]],
    vocab_list: List[str],
    vocab_bb: torch.Tensor,
    concept2bb: Dict[str, int],
    c: torch.Tensor,
    hyperbolic: bool = True,
    manifold=None,
    tag: str = "",
    max_eval: int = 2000,
    device: torch.device = torch.device("cpu"),
) -> Dict[str, float]:
    """KG link prediction with per-relation curvature when manifold is provided."""
    encoder.eval()
    rel_maps.eval()

    all_z = []
    with torch.no_grad():
        for i in range(0, len(vocab_list), 1024):
            batch_idx = torch.tensor(
                [concept2bb[c_name] for c_name in vocab_list[i:i + 1024]],
                device=device, dtype=torch.long
            )
            z_b = encoder.project(vocab_bb[batch_idx], c=c)
            all_z.append(z_b)
    all_z = torch.cat(all_z, dim=0)
    concept2idx = {c_name: i for i, c_name in enumerate(vocab_list)}

    ranks = []
    for h, r, t in tqdm(test_triples[:max_eval], desc=f"LinkPred {tag}", leave=False):
        if h not in concept2idx or t not in concept2idx:
            continue
        r_idx_val = REL2IDX[r]
        r_idx = torch.tensor([r_idx_val], device=device)

        c_r = manifold.c_rel(r_idx_val) if (manifold is not None and hyperbolic) else c

        z_h = all_z[concept2idx[h]].unsqueeze(0)
        with torch.no_grad():
            z_pred = rel_maps(z_h, r_idx, c=c_r)
            if hyperbolic and c_r.item() > EPS:
                dists = poincare_cdist(z_pred, all_z, c_r).squeeze(0)
            else:
                dists = eucl_cdist(z_pred, all_z).squeeze(0)
        rank = (dists < dists[concept2idx[t]]).sum().item() + 1
        ranks.append(rank)

    if not ranks:
        return {"MRR": 0, "Hits@1": 0, "Hits@10": 0, "n": 0}
    ranks = np.array(ranks)
    return {
        "MRR": float(np.mean(1.0 / ranks)),
        "Hits@1": float(np.mean(ranks <= 1)),
        "Hits@10": float(np.mean(ranks <= 10)),
        "n": len(ranks),
    }


def evaluate_crossmodal(
    vis_encoder,
    text_z: torch.Tensor,
    precomp_clip: torch.Tensor,
    precomp_lbl: torch.Tensor,
    c: torch.Tensor,
    hyperbolic: bool = True,
    tag: str = "",
    max_images: int = 2000,
    device: torch.device = torch.device("cpu"),
) -> Dict[str, float]:
    """Cross-modal retrieval: Image -> Text on CIFAR-100 test set."""
    vis_encoder.eval()

    cdist_fn = poincare_cdist if (hyperbolic and c.item() > EPS) else eucl_cdist

    n_eval = min(max_images, precomp_clip.shape[0])
    perm = torch.randperm(precomp_clip.shape[0])[:n_eval]

    with torch.no_grad():
        z_img = vis_encoder.project(precomp_clip[perm], c=c)
        labels = precomp_lbl[perm]

    ranks = []
    with torch.no_grad():
        if hyperbolic and c.item() > EPS:
            dists = cdist_fn(z_img, text_z, c)
        else:
            dists = cdist_fn(z_img, text_z)

        for i in range(n_eval):
            target = labels[i].item()
            rank = (dists[i] < dists[i, target]).sum().item() + 1
            ranks.append(rank)

    ranks = np.array(ranks)
    return {
        "R@1": float(np.mean(ranks <= 1)),
        "R@5": float(np.mean(ranks <= 5)),
        "R@10": float(np.mean(ranks <= 10)),
        "MedR": float(np.median(ranks)),
    }


def evaluate_hierarchy(
    vis_encoder,
    text_encoder,
    fine_bb: torch.Tensor,
    coarse_bb: torch.Tensor,
    fine2coarse: torch.Tensor,
    c: torch.Tensor,
    hyperbolic: bool = True,
    tag: str = "",
) -> Dict[str, object]:
    """
    Hierarchy depth test: fine-class embeddings should be deeper than coarse.
    """
    vis_encoder.eval()
    text_encoder.eval()

    if hyperbolic and c.item() > EPS:
        dist0_fn = lambda z: dist0(z, c)
    else:
        dist0_fn = lambda z: z.norm(dim=-1)

    with torch.no_grad():
        fine_z = text_encoder.project(fine_bb, c=c)
        coarse_z = text_encoder.project(coarse_bb, c=c)

    correct = 0
    total = 0
    depth_diffs = []
    for fine_idx in range(fine_z.shape[0]):
        coarse_idx = fine2coarse[fine_idx].item() if isinstance(fine2coarse, torch.Tensor) else fine2coarse[fine_idx]
        d_fine = dist0_fn(fine_z[fine_idx].unsqueeze(0)).item()
        d_coarse = dist0_fn(coarse_z[coarse_idx].unsqueeze(0)).item()
        if d_fine > d_coarse:
            correct += 1
        depth_diffs.append(d_fine - d_coarse)
        total += 1

    return {
        "accuracy": correct / max(total, 1),
        "mean_diff": float(np.mean(depth_diffs)) if depth_diffs else 0.0,
        "fine_depths": [dist0_fn(fine_z[i].unsqueeze(0)).item() for i in range(fine_z.shape[0])],
        "coarse_depths": [dist0_fn(coarse_z[i].unsqueeze(0)).item() for i in range(coarse_z.shape[0])],
    }


## Configuration

In [ ]:
import time as _time, gc, copy as _copy

CKPT_DIR = '/kaggle/working/checkpoints'
DATA_ROOT = '/kaggle/working/data'
import os
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(DATA_ROOT, exist_ok=True)

cfg = USMConfig(validation_mode=False)
cfg.ckpt_dir = CKPT_DIR

torch.manual_seed(cfg.seed)
random.seed(cfg.seed)
np.random.seed(cfg.seed)

print(f'Base config: d={cfg.d}, batch={cfg.batch_size}, GPUs={cfg.n_gpus}')
if torch.cuda.is_available():
    for i in range(cfg.n_gpus):
        props = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)} ({props.total_memory / 1e9:.1f} GB)')
print(f'Training device:  {cfg.device}')
print(f'Backbone device:  {cfg.backbone_device}')


## 1. Load Data

In [ ]:
print('--- ConceptNet ---')
triples = load_conceptnet(max_triples=cfg.max_cn_triples)

print('\n--- Cross-lingual ---')
cl_pairs = load_crosslingual(max_per_lang=cfg.max_cl_per_lang)

print('\n--- SNLI ---')
snli_pairs = load_snli(max_pairs=cfg.max_snli_pairs)

vocab_set = set()
for h, _, t in triples:
    vocab_set.add(h)
    vocab_set.add(t)
vocab_set.update(CIFAR100_FINE)
vocab_set.update(CIFAR100_COARSE)
vocab_list = sorted(vocab_set)
concept2bb = {c: i for i, c in enumerate(vocab_list)}

print(f'\nVocabulary: {len(vocab_list):,} concepts')
print(f'Triples:   {len(triples):,}')
print(f'CL pairs:  {len(cl_pairs):,}')
print(f'SNLI:      {len(snli_pairs):,}')

random.shuffle(triples)
n_test = max(500, len(triples) // 20)
test_triples = triples[:n_test]
train_triples = triples[n_test:]
print(f'Train: {len(train_triples):,}  |  Test: {len(test_triples):,}')


## 2. Pre-cache Backbone Embeddings

Text backbone (MiniLM, 384-dim) and CLIP (512-dim) are frozen — their outputs are dimension-independent and reused for **all** sweep runs.

In [ ]:
_run_start = _time.time()

_tmp_manifold = LearnablePoincareBall(
    c_init=cfg.c_init, c_min=cfg.c_min, c_max=cfg.c_max,
    learnable=False, n_relations=cfg.n_relations,
).to(cfg.device)
_tmp_encoder = ConceptEncoder(
    _tmp_manifold, d_out=cfg.d, backbone=cfg.text_backbone,
    hyperbolic=True, device=cfg.device, backbone_device=cfg.backbone_device,
).to(cfg.device)

print('Pre-caching text backbone embeddings (384-dim, reused for all dims)...')
vocab_bb = _precache_text_embeddings(_tmp_encoder, vocab_list, batch_size=cfg.encode_batch, device=cfg.device)
print(f'  vocab_bb: {vocab_bb.shape}')

if cl_pairs:
    cl_src_texts, cl_tgt_texts = zip(*cl_pairs)
    cl_bb_src = _precache_text_embeddings(_tmp_encoder, list(cl_src_texts), batch_size=cfg.encode_batch, device=cfg.device)
    cl_bb_tgt = _precache_text_embeddings(_tmp_encoder, list(cl_tgt_texts), batch_size=cfg.encode_batch, device=cfg.device)
    print(f'  cl_bb: {cl_bb_src.shape}')
else:
    cl_bb_src = cl_bb_tgt = None

if snli_pairs:
    snli_a, snli_b, snli_labels = zip(*snli_pairs)
    snli_bb_a = _precache_text_embeddings(_tmp_encoder, list(snli_a), batch_size=cfg.encode_batch, device=cfg.device)
    snli_bb_b = _precache_text_embeddings(_tmp_encoder, list(snli_b), batch_size=cfg.encode_batch, device=cfg.device)
    snli_labels_t = torch.tensor(snli_labels, device=cfg.device)
    print(f'  snli_bb: {snli_bb_a.shape}')
else:
    snli_bb_a = snli_bb_b = snli_labels_t = None

fine_bb = _tmp_encoder.encode_backbone(CIFAR100_FINE).to(cfg.device)
coarse_bb = _tmp_encoder.encode_backbone(CIFAR100_COARSE).to(cfg.device)
fine2coarse_tensor = torch.tensor(
    [CIFAR100_FINE2COARSE[i] for i in range(len(CIFAR100_FINE))],
    device=cfg.device,
)
print(f'  fine_bb: {fine_bb.shape}  coarse_bb: {coarse_bb.shape}')

del _tmp_encoder, _tmp_manifold
gc.collect()
torch.cuda.empty_cache()

print(f'\nPre-cache time: {_time.time() - _run_start:.1f}s')
if torch.cuda.is_available():
    for i in range(cfg.n_gpus):
        alloc = torch.cuda.memory_allocated(i) / 1e9
        total = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f'  GPU {i}: {alloc:.2f} / {total:.1f} GB')


## 3. Pre-cache CLIP (CIFAR-100)

In [ ]:
print('Loading CIFAR-100...')
cifar_train, cifar_test = load_cifar100(data_root=DATA_ROOT)
cifar_dl_train = DataLoader(cifar_train, batch_size=256, shuffle=False, num_workers=2)
cifar_dl_test = DataLoader(cifar_test, batch_size=256, shuffle=False, num_workers=2)

_tmp_manifold = LearnablePoincareBall(
    c_init=cfg.c_init, c_min=cfg.c_min, c_max=cfg.c_max,
    learnable=False, n_relations=cfg.n_relations,
).to(cfg.device)
_tmp_vis = VisionEncoder(
    _tmp_manifold, d_out=cfg.d, clip_model=cfg.clip_model,
    d_clip=cfg.d_clip, hyperbolic=True, device=cfg.device,
    backbone_device=cfg.backbone_device,
).to(cfg.device)

print('Pre-caching CLIP embeddings (512-dim, reused for all dims)...')
clip_train, clip_labels_train = _precache_clip_embeddings(
    _tmp_vis, cifar_dl_train, device=cfg.device, max_images=cfg.max_clip_images,
)
clip_test, clip_labels_test = _precache_clip_embeddings(
    _tmp_vis, cifar_dl_test, device=cfg.device, max_images=cfg.max_clip_images,
)
print(f'  CLIP train: {clip_train.shape}, test: {clip_test.shape}')

del _tmp_vis, _tmp_manifold
gc.collect()
torch.cuda.empty_cache()


## 4. Dimensionality Sweep

For each `d ∈ {32, 64, 128, 256, 512}`, trains:
- **Hyperbolic**: scheduled per-relation curvature warmup
- **Euclidean**: identical architecture, flat geometry (c=0)

35 P1 epochs + 20 P2 epochs per run. Both share the same pre-cached backbone embeddings.

In [ ]:
DIMS = [32, 64, 128, 256, 512]
SWEEP_P1_EPOCHS = 35
SWEEP_P2_EPOCHS = 20
EVAL_CAP = 1000

C_TARGETS_HYP = (0.05, 0.01, 0.03, 0.003, 0.003, 0.01)
C_WARMUP_START = 5
C_WARMUP_END = 28

all_results = {}
_sweep_start = _time.time()

for d_sweep in DIMS:
    for is_hyp in [True, False]:
        mode = "HYP" if is_hyp else "EUCL"
        key = f"d{d_sweep}_{mode}"

        print(f'\n{"=" * 65}')
        print(f'  {"HYPERBOLIC" if is_hyp else "EUCLIDEAN":>12s}  d = {d_sweep}')
        print(f'{"=" * 65}')

        torch.manual_seed(cfg.seed)
        random.seed(cfg.seed)
        np.random.seed(cfg.seed)

        cfg_s = _copy.deepcopy(cfg)
        cfg_s.d = d_sweep
        cfg_s.n_epochs_p1 = SWEEP_P1_EPOCHS
        cfg_s.n_epochs_p2 = SWEEP_P2_EPOCHS
        cfg_s.learnable_curvature = False

        if is_hyp:
            cfg_s.c_init = 0.001
            cfg_s.c_min = 0.0001
            cfg_s.c_targets = C_TARGETS_HYP
            cfg_s.c_warmup_start = C_WARMUP_START
            cfg_s.c_warmup_end = C_WARMUP_END
        else:
            cfg_s.c_init = 0.0
            cfg_s.c_min = 0.0
            cfg_s.c_max = 0.0
            cfg_s.c_targets = (0.0, 0.0, 0.0, 0.0, 0.0, 0.0)
            cfg_s.c_warmup_start = 999
            cfg_s.c_warmup_end = 999

        manifold_s = LearnablePoincareBall(
            c_init=cfg_s.c_init, c_min=cfg_s.c_min, c_max=cfg_s.c_max,
            learnable=False, n_relations=cfg_s.n_relations,
        ).to(cfg_s.device)

        encoder_s = ConceptEncoder(
            manifold_s, d_out=d_sweep, backbone=cfg_s.text_backbone,
            hyperbolic=is_hyp, device=cfg_s.device,
            backbone_device=cfg_s.backbone_device,
        ).to(cfg_s.device)

        vis_encoder_s = VisionEncoder(
            manifold_s, d_out=d_sweep, clip_model=cfg_s.clip_model,
            d_clip=cfg_s.d_clip, hyperbolic=is_hyp, device=cfg_s.device,
            backbone_device=cfg_s.backbone_device,
        ).to(cfg_s.device)

        comp_op_s = CompositionalOperator(manifold_s, d=d_sweep, hyperbolic=is_hyp).to(cfg_s.device)
        rel_maps_s = RelationMaps(manifold_s, d=d_sweep, hyperbolic=is_hyp).to(cfg_s.device)

        n_params = sum(p.numel() for p in [
            *manifold_s.parameters(), *encoder_s.parameters(), *vis_encoder_s.parameters(),
            *comp_op_s.parameters(), *rel_maps_s.parameters(),
        ] if p.requires_grad)
        print(f'  Trainable params: {n_params:,}')

        # ── Phase 1 ──────────────────────────────────────
        _t0 = _time.time()
        history_p1_s = train_phase1(
            cfg_s, manifold_s, encoder_s, comp_op_s, rel_maps_s,
            train_triples, cl_pairs, snli_pairs,
            vocab_list, vocab_bb, concept2bb,
            cl_bb_src=cl_bb_src, cl_bb_tgt=cl_bb_tgt,
            snli_bb_a=snli_bb_a, snli_bb_b=snli_bb_b,
            snli_labels_t=snli_labels_t,
        )
        p1_time = _time.time() - _t0

        c_eval = manifold_s.c
        lp = evaluate_link_prediction(
            encoder_s, rel_maps_s, test_triples, vocab_list,
            vocab_bb, concept2bb, c_eval,
            hyperbolic=is_hyp, manifold=manifold_s if is_hyp else None,
            tag=key, device=cfg_s.device, max_eval=EVAL_CAP,
        )

        # ── Phase 2 ──────────────────────────────────────
        _t1 = _time.time()
        history_p2_s = train_phase2(
            cfg_s, manifold_s, encoder_s, vis_encoder_s,
            clip_train, clip_labels_train,
            fine_bb, coarse_bb, fine2coarse_tensor,
        )
        p2_time = _time.time() - _t1

        with torch.no_grad():
            fine_z_s = encoder_s.project(fine_bb, c=c_eval)
        xm = evaluate_crossmodal(
            vis_encoder_s, fine_z_s, clip_test, clip_labels_test,
            c_eval, hyperbolic=is_hyp, tag=key, device=cfg_s.device,
        )
        hier = evaluate_hierarchy(
            vis_encoder_s, encoder_s, fine_bb, coarse_bb,
            fine2coarse_tensor, c_eval, hyperbolic=is_hyp, tag=key,
        )

        all_results[key] = {
            'd': d_sweep, 'mode': mode,
            'MRR': lp['MRR'], 'Hits@1': lp['Hits@1'], 'Hits@10': lp['Hits@10'],
            'R@1': xm['R@1'], 'R@5': xm['R@5'], 'R@10': xm['R@10'],
            'Hierarchy': hier['accuracy'], 'MeanDepth': hier['mean_diff'],
            'P1_loss': history_p1_s['loss_total'][-1],
            'P1_time': p1_time, 'P2_time': p2_time,
            'params': n_params,
        }

        print(f'\n  d={d_sweep} {mode}: MRR={lp["MRR"]:.4f}  H@10={lp["Hits@10"]:.4f}  '
              f'R@5={xm["R@5"]:.4f}  Hier={hier["accuracy"]:.0%}  '
              f'P1={p1_time/60:.1f}m  P2={p2_time/60:.1f}m')

        del manifold_s, encoder_s, vis_encoder_s, comp_op_s, rel_maps_s
        del history_p1_s, history_p2_s, fine_z_s
        gc.collect()
        torch.cuda.empty_cache()

_sweep_total = _time.time() - _sweep_start
print(f'\n\nSweep complete in {_sweep_total/60:.1f} min ({_sweep_total/3600:.1f} hrs)')


## 5. Results: Hyperbolic vs Euclidean

The thesis: **hyperbolic geometry achieves the same quality in far fewer dimensions**. At low d, the exponential volume of hyperbolic space gives it a decisive advantage for encoding hierarchical structure. At high d, both converge because Euclidean space has enough capacity.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

DIMS = [32, 64, 128, 256, 512]
METRICS = [
    ('MRR', 'MRR (Link Prediction)', True),
    ('Hits@10', 'Hits@10 (Link Prediction)', True),
    ('R@5', 'R@5 (Cross-modal Retrieval)', True),
    ('Hierarchy', 'Hierarchy Accuracy', True),
    ('MeanDepth', 'Mean Depth Diff (fine - coarse)', True),
    ('P1_loss', 'Final P1 Loss', False),
]

# ── Table ──
print('=' * 80)
print('  DIMENSIONALITY SWEEP — HYPERBOLIC vs EUCLIDEAN')
print('=' * 80)
header = f'{"d":>5s}  {"Geo":>5s}  {"Params":>10s}  {"MRR":>7s}  {"H@10":>7s}  {"R@5":>7s}  {"Hier":>7s}  {"Depth":>7s}  {"P1loss":>7s}'
print(header)
print('-' * 80)
for d in DIMS:
    for m in ['HYP', 'EUCL']:
        r = all_results[f'd{d}_{m}']
        print(f'{d:5d}  {m:>5s}  {r["params"]:10,d}  '
              f'{r["MRR"]:7.4f}  {r["Hits@10"]:7.4f}  '
              f'{r["R@5"]:7.4f}  {r["Hierarchy"]:7.2%}  '
              f'{r["MeanDepth"]:7.4f}  {r["P1_loss"]:7.4f}')
    delta_mrr = all_results[f'd{d}_HYP']['MRR'] - all_results[f'd{d}_EUCL']['MRR']
    marker = '<<< HYP wins' if delta_mrr > 0.005 else ('>>> EUCL wins' if delta_mrr < -0.005 else '    ~tie')
    print(f'       delta MRR = {delta_mrr:+.4f}  {marker}')

# ── Per-dim winner summary ──
print('\n' + '=' * 80)
print('  PER-DIMENSION WINNER (by MRR)')
print('=' * 80)
for d in DIMS:
    h = all_results[f'd{d}_HYP']['MRR']
    e = all_results[f'd{d}_EUCL']['MRR']
    winner = 'HYPERBOLIC' if h > e else 'EUCLIDEAN'
    pct = abs(h - e) / max(e, 1e-8) * 100
    print(f'  d={d:3d}:  {winner:12s}  (MRR {max(h,e):.4f} vs {min(h,e):.4f},  +{pct:.1f}%)')

# ── Plots ──
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for ax, (metric_key, title, higher_better) in zip(axes.flat, METRICS):
    hyp_vals = [all_results[f'd{d}_HYP'][metric_key] for d in DIMS]
    eucl_vals = [all_results[f'd{d}_EUCL'][metric_key] for d in DIMS]

    ax.plot(DIMS, hyp_vals, 'o-', linewidth=2, markersize=8,
            label='Hyperbolic', color='#2196F3')
    ax.plot(DIMS, eucl_vals, 's--', linewidth=2, markersize=8,
            label='Euclidean', color='#FF9800')

    for i, d in enumerate(DIMS):
        better = hyp_vals[i] > eucl_vals[i] if higher_better else hyp_vals[i] < eucl_vals[i]
        if better:
            ax.annotate('*', (d, hyp_vals[i]), fontsize=16, ha='center',
                        va='bottom', color='#2196F3', fontweight='bold')

    ax.set_xlabel('Embedding Dimension d')
    ax.set_ylabel(metric_key)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xscale('log', base=2)
    ax.set_xticks(DIMS)
    ax.set_xticklabels([str(d) for d in DIMS])
    ax.legend()
    ax.grid(alpha=0.3)

fig.suptitle('USM v2: Hyperbolic vs Euclidean across Dimensions',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
out_png = os.path.join(CKPT_DIR, 'usm_v2_dim_sweep.png')
plt.savefig(out_png, dpi=150, bbox_inches='tight')
plt.show()
print(f'\nPlot saved: {out_png}')


## 6. Save Results

In [ ]:
import json as _json

sweep_path = os.path.join(CKPT_DIR, 'usm_v2_sweep_results.json')
_serializable = {}
for k, v in all_results.items():
    _serializable[k] = {sk: float(sv) if isinstance(sv, (float, int)) else sv for sk, sv in v.items()}
with open(sweep_path, 'w') as f:
    _json.dump(_serializable, f, indent=2)
print(f'Results saved: {sweep_path}')

print(f'\nTotal sweep time: {(_time.time() - _run_start)/3600:.1f} hours')
